# TF-IDF + Logistic Regression + sublinear_tf=True

Experimento de classificação da clareza das respostas utilizando TF-IDF com ponderação sublinear da frequência dos termos e Regressão Logística.

In [4]:
import pandas as pd
import joblib

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, accuracy_score

In [5]:
!git clone https://github.com/juliaduboisas/plnEp1.git
%cd plnEp1
!ls

Cloning into 'plnEp1'...
remote: Enumerating objects: 157, done.
remote: Counting objects: 100% (157/157), done.
remote: Compressing objects: 100% (121/121), done.
remote: Total 157 (delta 69), reused 107 (delta 30), pack-reused 0 (from 0)
Receiving objects: 100% (157/157), 17.18 MiB | 11.50 MiB/s, done.
Resolving deltas: 100% (69/69), done.
/content/plnEp1/plnEp1
baseline  data	modelo_final  modelos_testados	README.md  resultados


## 1. Carregamento dos dados

In [6]:
df = pd.read_csv("data/train.csv")

Y = df["clarity"]

print(df.shape)
df.head()

(20092, 2)


,resp_text,clarity
0,"Prezado(a) Senhor(a), Esclarecemos que o Se...",c5
1,"Prezada cidadã, As informações sobre óbitos ...",c1
2,"Prezado Senhor Julio, A Ouvidoria-Geral da P...",c1
3,"Prezado(a) Senhor(a), Esclarecemos que o Se...",c234
4,"Senhor, O Serviço de Informações ao Cidadão d...",c234


## 2. Separação dos dados

In [7]:
x_train_text, x_test_text, y_train, y_test = train_test_split(
    df["resp_text"],
    Y,
    test_size=0.2,
    random_state=123,
    stratify=Y
)

print("Treino:", x_train_text.shape)
print("Teste:", x_test_text.shape)

Treino: (16073,)
Teste: (4019,)


## 3. Vetorização TF-IDF

Neste experimento, utilizamos `sublinear_tf=True`, que aplica uma ponderação sublinear à frequência dos termos.

In [8]:
vect = TfidfVectorizer(
    analyzer="word",
    sublinear_tf=True,
    ngram_range=(1, 3)
)

x_train = vect.fit_transform(x_train_text)
x_test = vect.transform(x_test_text)

print("Treino TF-IDF:", x_train.shape)
print("Teste TF-IDF:", x_test.shape)

Treino TF-IDF: (16073, 1241648)
Teste TF-IDF: (4019, 1241648)


## 4. Regressão Logística

In [9]:
clf = LogisticRegression(
    C=1,
    class_weight="balanced",
    max_iter=1000
)

clf.fit(
    x_train,
    y_train
)

LogisticRegression(C=1, class_weight='balanced', max_iter=1000)

## 5. Avaliação

In [10]:
predicted = clf.predict(x_test)

score = f1_score(
    y_test,
    predicted,
    average="macro"
)

accuracy = accuracy_score(
    y_test,
    predicted
)

print(f"Macro F1: {score:.4f}")
print(f"Accuracy: {accuracy:.4f}")

Macro F1: 0.4772
Accuracy: 0.4772


In [11]:
from sklearn.metrics import classification_report

print(
    classification_report(
        y_test,
        predicted
    )
)

              precision    recall  f1-score   support

          c1       0.50      0.52      0.51      1269
        c234       0.41      0.39      0.40      1371
          c5       0.52      0.53      0.52      1379

    accuracy                           0.48      4019
   macro avg       0.48      0.48      0.48      4019
weighted avg       0.48      0.48      0.48      4019



In [12]:
# ============================================================
# VALIDAÇÃO CRUZADA — TF-IDF + Logistic Regression
# Melhor configuração encontrada
# ============================================================

import pandas as pd
import numpy as np

from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_validate


# ------------------------------------------------------------
# 1. Dados
# ------------------------------------------------------------

df = pd.read_csv("../data/train.csv")

X = df["resp_text"]
y = df["clarity"]


# ------------------------------------------------------------
# 2. Melhor configuração encontrada
# ------------------------------------------------------------

pipeline = Pipeline([
    (
        "tfidf",
        TfidfVectorizer(
            analyzer="word",
            sublinear_tf=True,
            ngram_range=(1, 3)
        )
    ),
    (
        "logreg",
        LogisticRegression(
            C=1.0,
            class_weight="balanced"
        )
    )
])


# ------------------------------------------------------------
# 3. Cross-validation estratificada
# ------------------------------------------------------------

cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=123
)


# ------------------------------------------------------------
# 4. Avaliação
# ------------------------------------------------------------

results = cross_validate(
    pipeline,
    X,
    y,
    cv=cv,
    scoring={
        "f1_macro": "f1_macro",
        "accuracy": "accuracy"
    },
    return_train_score=True,
    n_jobs=-1
)


# ------------------------------------------------------------
# 5. Resultados por fold
# ------------------------------------------------------------

print("Resultados por fold:\n")

for i in range(5):
    print(
        f"Fold {i+1}: "
        f"F1 Macro = {results['test_f1_macro'][i]:.4f} | "
        f"Accuracy = {results['test_accuracy'][i]:.4f}"
    )


# ------------------------------------------------------------
# 6. Média e desvio padrão
# ------------------------------------------------------------

print("\nResultado da validação cruzada:")

print(
    f"Macro F1: "
    f"{results['test_f1_macro'].mean():.4f} "
    f"± {results['test_f1_macro'].std():.4f}"
)

print(
    f"Accuracy: "
    f"{results['test_accuracy'].mean():.4f} "
    f"± {results['test_accuracy'].std():.4f}"
)

print("\nTreino:")
print(
    f"Macro F1: "
    f"{results['train_f1_macro'].mean():.4f} "
    f"± {results['train_f1_macro'].std():.4f}"
)

print(
    f"Accuracy: "
    f"{results['train_accuracy'].mean():.4f} "
    f"± {results['train_accuracy'].std():.4f}"
)

Resultados por fold:

Fold 1: F1 Macro = 0.4582 | Accuracy = 0.4596
Fold 2: F1 Macro = 0.4669 | Accuracy = 0.4680
Fold 3: F1 Macro = 0.4650 | Accuracy = 0.4652
Fold 4: F1 Macro = 0.4574 | Accuracy = 0.4577
Fold 5: F1 Macro = 0.4669 | Accuracy = 0.4671

Resultado da validação cruzada:
Macro F1: 0.4629 ± 0.0042
Accuracy: 0.4635 ± 0.0041

Treino:
Macro F1: 0.8575 ± 0.0014
Accuracy: 0.8576 ± 0.0014


## 6. Treinamento do modelo final com todos os dados

In [13]:
# ============================================================
# TREINAMENTO DO MODELO FINAL COM TODOS OS DADOS ROTULADOS
# ============================================================

# Dados completos de treinamento
X_full = df["resp_text"]
y_full = df["clarity"]

# Novo TF-IDF
vect_final = TfidfVectorizer(
    analyzer="word",
    sublinear_tf=True,
    ngram_range=(1, 3)
)

# Ajustar o TF-IDF usando TODOS os dados rotulados
X_full_tfidf = vect_final.fit_transform(X_full)

print("TF-IDF final:", X_full_tfidf.shape)

# Novo classificador
clf_final = LogisticRegression(
    C=1.0,
    class_weight="balanced",
    max_iter=1000
)

# Treinar usando TODOS os dados rotulados
clf_final.fit(X_full_tfidf, y_full)

print("Modelo final treinado com todos os dados.")

TF-IDF final: (20092, 1473992)
Modelo final treinado com todos os dados.


## 7. Rotulação o conjunto de teste com o modelo final

In [14]:
# ============================================================
# CLASSIFICAÇÃO DO CONJUNTO DE TESTE EXTERNO
# ============================================================

test_df = pd.read_excel("data/test1.xlsx")

X_new = test_df["resp_text"]

# Aplicar o TF-IDF final aos novos textos
X_new_tfidf = vect_final.transform(X_new)

# Fazer as previsões com o modelo final
predicted_test = clf_final.predict(X_new_tfidf)

# Adicionar as previsões
test_df["clarity"] = predicted_test

# Conferir resultado
print(test_df["clarity"].value_counts())
print(test_df.head())

clarity
c5      315
c234    301
c1      284
Name: count, dtype: int64
                                           resp_text clarity
0   Prezada Hellen,    Dentre as pesquisas domici...    c234
1   Senhor Francisco,  O Serviço de Informações a...    c234
2   Prezada Srª. Raylane, Em resposta a sua solic...      c5
3   Prezada Senhora,  De acordo com o art.13 do D...      c1
4   Prezado(a) Senhor(a),   Em atendimento ao ped...    c234


In [16]:
# Salva o arquivo rotulado
test_df.to_excel("resultados/test1_rotulado.xlsx", index=False)

In [17]:
# Instala a biblioteca eli5 para interpretação do modelo
!pip install eli5
import eli5

## 8. Interpretação Global do Modelo
Nesta seção, analisamos quais palavras ou n-gramas o classificador LogisticRegression considera mais importantes para prever cada classe (c1, c234, c5).

In [18]:
# Mostra os 30 termos com maiores pesos globais para cada classe
# Usamos o clf_final e vect_final que foram treinados com todos os dados
eli5.show_weights(clf_final, vec=vect_final, top=30)

## 9. Carregamento do Gabarito e Previsões
Vamos carregar o arquivo com as respostas reais do teste 1 e gerar as previsões do nosso modelo final para podermos analisar os acertos e os erros locais.

In [21]:
# Carrega o arquivo com o gabarito real fornecido
df_gabarito = pd.read_excel("/content/test1-rotulado-para-teste-estatistico.xlsx")

# Extrai os textos e os rótulos reais
X_teste_gab = df_gabarito["resp_text"]
y_real_gab = df_gabarito["clarity"]

# Transforma os textos e faz a predição com o modelo final
X_teste_gab_tfidf = vect_final.transform(X_teste_gab)
y_pred_gab = clf_final.predict(X_teste_gab_tfidf)

# Salva as predições no dataframe do gabarito para comparação
df_gabarito["predicao_modelo"] = y_pred_gab

print("Predições realizadas com sucesso no gabarito.")

Predições realizadas com sucesso no gabarito.


## 10. Interpretação Local (Exemplo Único)
Aqui analisamos o peso de cada palavra na decisão do modelo para um documento específico. O verde indica palavras que aumentaram a probabilidade da classe, e o vermelho indica palavras que a diminuíram.

In [22]:
# Selecione um índice aleatório ou específico para analisar
indice_exemplo = 0
texto_exemplo = df_gabarito.loc[indice_exemplo, "resp_text"]
classe_real = df_gabarito.loc[indice_exemplo, "clarity"]

print(f"Classe Real (Gabarito): {classe_real}")
print(f"Classe Predita (Modelo): {df_gabarito.loc[indice_exemplo, 'predicao_modelo']}")

# Mostra o texto destacado com os pesos locais
eli5.show_prediction(clf_final, texto_exemplo, vec=vect_final)

Classe Real (Gabarito): c5
Classe Predita (Modelo): c234


## 11. Análise de Erros (Exportação)
Filtramos os casos onde a predição do modelo diverge do gabarito para investigação qualitativa das limitações do TF-IDF + Regressão Logística.

In [23]:
# Filtra apenas as linhas onde houve divergência
mascara_erros = df_gabarito["clarity"] != df_gabarito["predicao_modelo"]
df_erros = df_gabarito[mascara_erros].copy()

print(f"Total de classificações corretas: {len(df_gabarito) - len(df_erros)}")
print(f"Total de erros: {len(df_erros)}")

# Seleciona as colunas de interesse para facilitar a leitura
df_erros_simplificado = df_erros[["resp_text", "clarity", "predicao_modelo"]]

# Exporta os erros para um CSV para ler externamente
df_erros_simplificado.to_csv("analise_de_erros_teste1.csv", index=False)

# Visualiza os 5 primeiros erros no notebook
df_erros_simplificado.head()

Total de classificações corretas: 447
Total de erros: 453


,resp_text,clarity,predicao_modelo
0,"Prezada Hellen, Dentre as pesquisas domici...",c5,c234
1,"Senhor Francisco, O Serviço de Informações a...",c1,c234
6,A informação solicitada encontra-se na página...,c5,c1
7,"Prezado (a) Cidadão (ã), 1.Conforme solicit...",c1,c234
9,"Prezado Ssenhor, Em atendimento a sua soli...",c234,c1
